# PHYS 338 — Physique Statistique — Homework 1
## Information, Probabilités, Entropie

**Nom :** ...
**Date :** ...


### Dépendances
- Python ≥ 3.9
- `numpy`, `matplotlib`, `scipy`
- modules standard : `os`, `zlib`, `bz2`, `lzma`, `collections`

In [1]:
import os
import zlib, bz2, lzma
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=338)  # graine fixée pour la reproductibilité

---
# Exercice 1 — Entropie et compression des données

## Part I — Surprise, incertitude et information

Variable aléatoire $X$ à valeurs dans un alphabet $\chi$ de $L$ lettres, $P(X = x_i) = p_i$.

$\chi = \{A, B, C, D\}$ avec $p_A = 1/2,\ p_B = 1/4,\ p_C = 1/8,\ p_D = 1/8$.

### Q1 — Incertitude $H[X]$
Surprise d'un tirage : $\log_2 (1/p_i)$. L'incertitude $H[X]$ est la surprise moyenne.
Comment s'écrit l'incertitude en général ? Que vaut-elle dans notre exemple ?

**Réponse :**

**Cas général**:  
La surprise associée au tirage $X = x_i$ est $\log_2 \frac{1}{p_i}$. L'incertitude $H[X]$ est la moyenne de la surprise étant pondérée par sa probabilité $p_i$ :

$$
H[X] = \sum_{i=1}^{L} p_i \log_2 \frac{1}{p_i} = -\sum_{i=1}^{L} p_i \log_2 p_i
$$


**Dans notre exemple**:


$$
H[X] = \frac{1}{2} + \frac{1}{2} + \frac{3}{8} + \frac{3}{8} = \frac{7}{4}.
$$

### Q2 — Bornes de $H[X]$
Montrer que $0 \le H[X] \le \log_2 L$ ; que $0$ est atteint ssi une seule lettre a une probabilité non nulle ;
que $\log_2 L$ est atteint ssi toutes les lettres sont équiprobables.

**Réponse :**


**Borne inférieure : $H[X] \ge 0$.** Pour tout $i$, $0 \le p_i \le 1$, donc $\log_2 \frac{1}{p_i} \ge 0$ et chaque terme vérifie $p_i \log_2 \frac{1}{p_i} \ge 0$. Une somme de termes positifs est positive :

$$
H[X] = \sum_{i=1}^{L} p_i \log_2 \frac{1}{p_i} \ge 0.
$$

**Cas d'égalité $H[X] = 0$.** Une somme de termes positifs est nulle si et seulement si chaque terme est nul. Or $p_i \log_2 \frac{1}{p_i} = 0$ si et seulement si $p_i = 0$ ou $p_i = 1$. Comme $\sum_i p_i = 1$, cela impose qu'une seule lettre ait une probabilité non nulle. Réciproquement, si une seule lettre a $p_j = 1$ et toutes les autres $p_i = 0$, tous les termes sont nuls et $H[X] = 0$.

**Borne supérieure : $H[X] \le \log_2 L$.** On utilise l'inégalité $\ln x \le x - 1$, valable pour tout $x > 0$, avec égalité si et seulement si $x = 1$. En notant $S = \{i : p_i > 0\}$ l'ensemble des lettres de probabilité non nulle :

$$
H[X] - \log_2 L = \sum_{i \in S} p_i \log_2 \frac{1}{L\,p_i} = \frac{1}{\ln 2} \sum_{i \in S} p_i \ln \frac{1}{L\,p_i}
\le \frac{1}{\ln 2} \sum_{i \in S} p_i \left( \frac{1}{L\,p_i} - 1 \right).
$$

Or

$$
\sum_{i \in S} p_i \left( \frac{1}{L\,p_i} - 1 \right) = \frac{|S|}{L} - \sum_{i \in S} p_i = \frac{|S|}{L} - 1 \le 0,
$$

car $|S| \le L$. D'où $H[X] \le \log_2 L$.

**Cas d'égalité $H[X] = \log_2 L$.** Les deux inégalités doivent être des égalités :
- $\ln x = x - 1$ impose $\frac{1}{L\,p_i} = 1$, soit $p_i = \frac{1}{L}$ pour tout $i \in S$ ;
- $\frac{|S|}{L} = 1$ impose $|S| = L$, c'est-à-dire que toutes les lettres ont une probabilité non nulle.

Donc $p_i = \frac{1}{L}$ pour tout $i$ : **toutes les lettres sont équiprobables**. Réciproquement, dans ce cas $H[X] = \sum_{i=1}^{L} \frac{1}{L} \log_2 L = \log_2 L$.

**Interprétation.** L'incertitude est nulle quand le résultat est certain, et maximale quand on n'a aucune raison de préférer une lettre à une autre. Dans notre exemple, $H[X] = 1{,}75 < \log_2 4 = 2$ bits, car la distribution n'est pas uniforme.

### Q3 — Information manquante
L'information manquante sur la valeur secrète $x_i$ est $\log_2(1/p_i)$.
Quelle est sa valeur moyenne en général ? Que vaut-elle dans notre exemple ?

**Réponse :**

**Cas général.** Si la valeur mesurée est $x_i$, l'information qu'il vous manque est $\log_2 \frac{1}{p_i}$. Cette valeur $x_i$ apparaît avec probabilité $p_i$, donc l'information manquante moyenne est l'espérance :

$$
\langle I \rangle = \sum_{i=1}^{L} p_i \log_2 \frac{1}{p_i} = -\sum_{i=1}^{L} p_i \log_2 p_i = H[X]
$$

C'est exactement la même expression que l'incertitude $H[X]$ de la Q1. Les deux points de vue décrivent la même quantité, vue avant ou après le tirage :
- *avant* le tirage, $H[X]$ mesure votre incertitude moyenne, c'est-à-dire la surprise que vous aurez en moyenne ;
- *après* le tirage, $H[X]$ mesure l'information moyenne qu'il vous faut recevoir pour connaître le résultat.

L'information apportée en révélant $x_i$ est donc précisément ce qui lève votre incertitude : l'entropie de Shannon s'interprète indifféremment comme une mesure d'**incertitude** ou d'**information manquante**.

**Dans notre exemple :**

$$
\langle I \rangle = \frac{1}{2}\cdot 1 + \frac{1}{4}\cdot 2 + \frac{1}{8}\cdot 3 + \frac{1}{8}\cdot 3 = \frac{7}{4} = 1{,}75 \text{ bits}.
$$

À comparer avec $\log_2 4 = 2$ bits pour quatre lettres équiprobables : connaître les probabilités vous donne déjà une partie de l'information, donc il vous en manque moins en moyenne.

## Part II — Entropie de Shannon et questions OUI/NON

### Q1 — Stratégie de questions pour {A, B, C, D}
Combien de questions OUI/NON faut-il poser en moyenne pour trouver la lettre tirée ?
Comparer avec l'entropie de l'alphabet.

**Réponse :**


**Stratégie.** Pour trouver la lettre le plus vite possible en moyenne, chaque question doit apporter le maximum d'information. On découpe donc à chaque étape les lettres restantes en deux groupes de probabilités égales, et on demande si la lettre appartient au premier groupe. Chaque réponse est alors équiprobable et apporte exactement 1 bit d'information.

Avec $p_A = 1/2$, $p_B = 1/4$, $p_C = p_D = 1/8$ :

1. « La lettre est-elle dans $\{A\}$ ? » : $\{A\}$ contre $\{B, C, D\}$, soit $1/2$ contre $1/2$. Si OUI, on a trouvé en 1 question.
2. Sinon, « La lettre est-elle dans $\{B\}$ ? » : $\{B\}$ contre $\{C, D\}$, soit $1/4$ contre $1/4$. Si OUI, on a trouvé en 2 questions.
3. Sinon, « La lettre est-elle dans $\{C\}$ ? » : la réponse donne C ou D en 3 questions.


**Nombre moyen de questions.**

$$
\langle n \rangle = \sum_{i} p_i \, n_i = \frac{1}{2}\cdot 1 + \frac{1}{4}\cdot 2 + \frac{1}{8}\cdot 3 + \frac{1}{8}\cdot 3 = \frac{7}{4} = H[X]
$$


On obtient donc $\langle n \rangle = H[X]$ : le nombre moyen de questions est exactement égal à l'entropie. On remarque aussi que chaque lettre est trouvée en $n_i = \log_2 1/p_i$ questions, c'est-à-dire que le nombre de questions pour une lettre est égal à sa surprise.



### Q2 — Alphabet de $2^b$ lettres équiprobables
Combien de questions faut-il poser en moyenne ? Comparer avec l'entropie du système.

**Réponse :**

**Stratégie.** Toutes les lettres ont la même probabilité $p_i = 1/2^b$. Pour couper en deux groupes équiprobables, il suffit de couper l'ensemble des lettres restantes **en deux moitiés de même taille** (recherche dichotomique) :

- avant la 1re question : $2^b$ lettres possibles ;
- après la 1re question : $2^{b-1}$ lettres ;
- après la $k$-ième question : $2^{b-k}$ lettres.

La lettre est identifiée quand il ne reste plus qu'une seule lettre, soit $2^{b-k} = 1$, donc après $k = b$ questions.

**Nombre moyen de questions.** Chaque lettre demande exactement $b$ questions, quelle que soit la lettre tirée :

$$
\langle n \rangle = \sum_{i=1}^{2^b} \frac{1}{2^b} \cdot b = b.
$$

**Comparaison avec l'entropie.** Pour une distribution uniforme sur $L = 2^b$ lettres :

$$
H[X] = \sum_{i=1}^{2^b} \frac{1}{2^b} \log_2 2^b = \log_2 2^b = b .
$$

On retrouve $\langle n \rangle = H[X] = b$. C'est aussi la valeur maximale $\log_2 L$ de l'entropie.


### Q3 — Théorème de Shannon (1948)
Alphabet général de $A$ lettres de probabilités $p_i$. Comment généraliser la stratégie ?
Combien de questions en moyenne ? (On suppose qu'on peut toujours couper en deux groupes de probabilité 1/2.)

**Réponse :**

**Généralisation de la stratégie.** On considère $A$ lettres $x_i$ de probabilités $p_i$. À chaque étape, on partage les lettres restantes en deux groupes de même probabilité , et on demande si la lettre tirée appartient au premier groupe. Chaque réponse OUI/NON est équiprobable et apporte donc exactement 1 bit d'information.

**Nombre de questions pour une lettre donnée.** Au départ, l'ensemble des lettres a une probabilité totale de $1$. Chaque question divise par deux la probabilité du groupe dans lequel se trouve la lettre :

$$
\text{après } k \text{ questions, le groupe restant a une probabilité } 2^{-k}.
$$

La lettre $x_i$ est identifiée lorsque le groupe restant se réduit à $\{x_i\}$, c'est-à-dire lorsque sa probabilité vaut $p_i$ :

$$
2^{-n_i} = p_i \quad \Longrightarrow \quad n_i = \log_2 \frac{1}{p_i}.
$$

Le nombre de questions nécessaires pour trouver $x_i$ est donc exactement sa **surprise**.

**Nombre moyen de questions.**

$$
\langle n \rangle = \sum_{i=1}^{A} p_i \, n_i = \sum_{i=1}^{A} p_i \log_2 \frac{1}{p_i} = H[X].
$$


## Part III — Compression de données
Fichier de $N$ valeurs dans un alphabet de $L$ lettres : taille $N \log_2 L$ bits.

### Q1 — Compression par questions OUI/NON
On remplace chaque lettre par la suite de ses réponses (OUI = 1, NON = 0).
Taille du nouveau fichier ? Facteur de compression (taille avant / taille après) ?

**Réponse :**



### \* Q2 — Comparaison avec un compresseur usuel
Fichiers de $10^4$ caractères ASCII (8 bits) :
1. `uniform.txt` : parmi les 256 caractères
2. `half.txt` : parmi 128 caractères
3. `abcd.txt` : parmi ABCD (équiprobables)
4. `abcd2.txt` : parmi ABCD avec les probabilités de la partie II

> Placer les fichiers dans le même dossier que ce notebook (ou modifier `DATA_DIR`).

**Réponse :**


## Part IV — L'entropie de l'anglais

### \* Q1 — Rapport de compression de l'anglais
Fréquences des lettres : <http://en.wikipedia.org/wiki/Letter_frequencies>.
Calculer le rapport de compression de l'anglais (ignorer espaces, majuscules et caractères spéciaux).

**Réponse :**



### \* Q2 — Comparaison avec les meilleurs compresseurs
Voir <http://en.wikipedia.org/wiki/Hutter_Prize>. D'où vient la différence observée ?

**Réponse :**



### \* Q3 — Expérience de Shannon
Estimer (et commenter) l'entropie de l'anglais avec
<https://www.csfieldguide.org.nz/en/interactives/shannon-experiment/>.

**Réponse :**



---
# Exercice 2 — Estimation d'un taux de désintégration (loi de Poisson)

$P(X = k) = e^{-\lambda^*} \dfrac{(\lambda^*)^k}{k!}$, $k \in \mathbb{N}$. On prend $\lambda^* = 3$ pour les simulations.

In [ ]:
lambda_star = 3.0

### Q1 — Fonction génératrice des moments
a) Écrire $M_X(t) = \mathbb{E}[e^{tX}]$ comme une somme sur $k$.
b) Reconnaître une série usuelle et en déduire $M_X(t) = \exp\big(\lambda^*(e^t - 1)\big)$.
c) En déduire $\kappa_X(t) = \log M_X(t)$, puis $\mathbb{E}[X] = \kappa_X'(0)$ et $\mathrm{Var}(X) = \kappa_X''(0)$. Remarque ?

**Réponse :**

$$M_X(t)=\mathbb{E}[e^{tX}]=\sum_{k=0}^\infty e^{tk}e^{-\lambda^*} \frac{(\lambda^*)^k}{k!}=e^{-\lambda^*}\sum_{k=0}^\infty \frac{(\lambda^*e^t)^k}{k!}$$
On reconnait la série $\sum_{k=0}^\infty\frac{a^k}{k!}=e^a$ avec $a=\lambda^*e^t$ donc $$M_X(t)=e^{-\lambda^*}e^{\lambda^*e^t}=e^{\lambda^*(e^t-1)} \quad \text{et} \quad \kappa_X(t)=\log M_X(t)=\log e^{\lambda^*(e^t-1)}=\lambda^*(e^t-1).$$  
$\mathbb{E}[X]=\sum_{k=0}^\infty ke^{-\lambda^*}\frac{(\lambda^*)^k}{k!}=e^{-\lambda^*}\sum_{k=0}^\infty k\frac{(\lambda^*)^k}{k!}=e^{-\lambda^*}\sum_{k=1}^\infty \frac{(\lambda^*)^k}{(k-1)!}=e^{-\lambda^*}\sum_{k=0}^\infty \lambda^*\frac{(\lambda^*)^k}{k!}=\lambda^*$  
  
$\kappa'_X(t)=\lambda^*e^t \Rightarrow \kappa'_X(0)=\lambda^*=\mathbb{E}[X]$



### \* Q2 — Génération de données
Comment générer des variables de Poisson($\lambda$) ? Écrire une routine générant $N$ échantillons
(numpy, ou méthode de Knuth).

In [2]:
def poisson(lam, N):
    return np.random.poisson(lam, N)


**Réponse :**

*(à compléter)*

### Q3 — Estimateur $\hat\lambda = \frac{1}{N}\sum_i X_i$
Moyenne ? Variance ? Erreur typique ?

**Réponse :**

*(à compléter)*

### \* Q4 — Convergence numérique
Vérifier que $\hat\lambda \to \lambda^*$ quand $N$ augmente. Tracer la distribution empirique de $\hat\lambda$
pour différentes valeurs de $N$ (répéter l'expérience $M$ fois à $N$ fixé).

In [ ]:
N_valeurs = [10, 100, 1000]   # à ajuster
M = 10_000                    # nombre de répétitions à N fixé (à ajuster)

def estimations_lambda(lam, N, M):
    # TODO : renvoyer un tableau de M valeurs de lambda_hat
    pass

In [ ]:
# TODO : graphiques (histogrammes de lambda_hat pour chaque N, écart à lambda* en fonction de N, ...)

**Réponse :**

*(à compléter)*

### \* Q5 — Théorème central limite
Rappeler le TCL dans ce cadre et préciser la loi limite de $S_N = \sqrt{N}\,\hat\lambda_N$.
Qu'est-ce que cela implique pour la loi de $\hat\lambda_N$ ? Vérifier numériquement.

**Réponse :**

*(à compléter)*

In [ ]:
# TODO : histogrammes numériques comparés à la loi limite prédite

### Q6 — Grandes déviations
$P(\hat\lambda_N \approx s) \asymp e^{-N I(s)}$, avec (Cramér) $I(s) = \sup_{t \in \mathbb{R}} \{ ts - \kappa_X(t) \}$.
Calculer $I(s)$.

**Réponse :**

*(à compléter)*

### Q7 — Développement autour de la moyenne
Développer $I(s)$ au voisinage de $s = \lambda^*$. Montrer qu'on retrouve l'approximation gaussienne du TCL.
Commenter les déviations par rapport à la gaussienne.

**Réponse :**

*(à compléter)*

### \* Q8 — Vérification numérique
Pour plusieurs $N$, simuler un grand nombre d'expériences : comparer l'histogramme de $S_N$ à la gaussienne du TCL.
Jusqu'où l'approximation gaussienne est-elle valable ? Vérifier la loi des grandes déviations.

In [ ]:
def I_cramer(s, lam):
    # TODO : fonction de taux obtenue en Q6
    pass

def I_gauss(s, lam):
    # TODO : approximation quadratique obtenue en Q7
    pass

In [ ]:
# TODO : simulations pour plusieurs N (grand nombre d'expériences)

In [ ]:
# TODO : histogramme de S_N vs gaussienne du TCL (échelle log conseillée pour voir les queues)

In [ ]:
# TODO : comparer -(1/N) log P(lambda_hat ≈ s) empirique à I(s) et à l'approximation gaussienne

**Réponse :**

*(à compléter)*

---
## Conclusion
*(à compléter)*